# Cox survival analysis

* `mydir`: your own data
* `ostime_column`: the survival time of the data; it does not have to be OST, it can also be DST, FST, etc.
* `os`: survival status; it does not have to be OS, it can also be DS, FS, etc.

In [ ]:
from lifelines import CoxPHFitter
import pandas as pd
from onekey_algo.custom.components.comp1 import normalize_df
from sklearn.model_selection import train_test_split
from onekey_algo import get_param_in_cwd
from onekey_algo.custom.components.comp1 import fillna

event_col = get_param_in_cwd('event_col')
group_info = 'group'
task_type = 'Clinical_'
duration_col= get_param_in_cwd('duration_col')
data = pd.read_csv('data/clinical.csv', dtype={'ID': str})
data = data[data.columns[:-3]]
# data = normalize_df(data, not_norm='ID')
label_data = pd.read_csv(get_param_in_cwd('label_file'), dtype={'ID': str})
label_data['ID'] = label_data['ID'].map(lambda x: f"{x}.nii.gz" if not (f"{x}".endswith('.nii.gz') or  f"{x}".endswith('.nii')) else x)
data = pd.merge(data, label_data[['ID', event_col, duration_col, 'group']], on='ID', how='inner')
data

In [ ]:
import numpy as np
import onekey_algo.custom.components as okcomp
from collections import OrderedDict


train_data = data[(data[group_info] == 'train')]

# subsets = [s for s in label_data['group'].value_counts().index if s != 'train']
subsets = get_param_in_cwd('subsets')
val_datasets = OrderedDict()
for subset in subsets:
    val_data = data[data[group_info] == subset]
    val_datasets[subset] = val_data
    val_data.to_csv(f'features/{task_type}{subset}_features_norm.csv', index=False)

print(', '.join([f"{subset} sample count: {d_.shape}" for subset, d_ in val_datasets.items()]))

In [ ]:
import os
from onekey_algo.custom.components.survival import uni_cox

if os.path.exists(f'features/{task_type}features_unisel.csv') and False:
    train_data = pd.read_csv(f'features/{task_type}features_unisel.csv')
else:
    sel_features, uni_info = uni_cox(train_data, duration_col=duration_col, event_col=event_col,
                                     cols=[c for c in train_data.columns if c not in [event_col, duration_col, 'ID', 'group']], 
                                     pvalue_thres=0.05, verbose=True)
    train_data = train_data[['ID'] + sel_features + [event_col, duration_col, 'group']]
    data[['ID'] + sel_features + [event_col, duration_col, 'group']].to_csv(f'features/{task_type}features_unisel.csv', header=True, index=False)
train_data

## Cox overview

All the data necessary for Cox regression; there are 3 main quantities of interest
1. `Concordance`: c-index
2. `exp(coef)`: the HR for each feature, together with its corresponding 95% percentile.
3. `p`: indicates whether the feature is significant.

In [ ]:
from lifelines import CoxPHFitter

cph = CoxPHFitter(penalizer=0.5)
cph.fit(train_data[[c for c in train_data.columns if c not in ['ID', 'group']]], duration_col=duration_col, event_col=event_col)
cph.print_summary()

In [ ]:
# Multivariable Cox model using clinical variables selected a priori based on
# clinical relevance and prior literature.
# Define the candidate clinical variables.
# These variables will all be retained in the final multivariable results and CSV data
selected_features = ['Child_Pugh','Gender','Total_SLD_cm','Age','Diabetes','AFP','CA199','Lesion_Count','Hepatitis_C']

# If there are radiomics scores, just uncomment manually
# if 'Rad_Score' in train_data.columns:
#     selected_features.append('Rad_Score')

# Fit the multivariable Cox model on the training set.
from lifelines import CoxPHFitter
cph = CoxPHFitter()
# Extract training-set data from the raw data, avoiding the filtered train_data
train_data_full = data[data[group_info] == 'train']
# Fit using only the training set to ensure the scientific validity of the model parameters
cph.fit(train_data_full[['ID', duration_col, event_col] + selected_features], 
        duration_col=duration_col, event_col=event_col, cluster_col='ID')

# Merge the multivariable results with the univariable table.
multi_info = cph.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']].reset_index()
multi_info.columns = ['covariate', 'HR', 'lower 95%CI', 'upper 95%CI', 'p']

# Use a left join to ensure all variables in the univariable table are retained
i = pd.merge(uni_info, multi_info, on='covariate', how='left', suffixes=['_Uni', '_Multi'])

# Save the selected clinical features for all cohorts.
# Note: here we use the full dataset data and directly use the manually defined selected_features, without P-value filtering
# Ensure ID, event, time, and group label (group) are included
multi_data = data[['ID'] + selected_features + [event_col, duration_col, 'group']]
multi_data.to_csv(f'features/{task_type}features_mulsel.csv', header=True, index=False)

# Format the summary table.
# Handle P values: show '<0.05' when below 0.05, otherwise keep three decimals, leave null values empty
def p_format(x):
    if pd.isna(x) or x == '': return ""
    return '<0.05' if x < 0.05 else f"{x:.3f}"

i['p_Uni'] = i['p_Uni'].apply(p_format)
i['p_Multi'] = i['p_Multi'].apply(p_format)

# Standardize column names: clearly distinguish univariable (Uni) and multivariable (Multi)
i.columns = ['features_name', 'HR_Uni', 'lower_Uni', 'upper_Uni', 'p_Uni', 
             'HR_Multi', 'lower_Multi', 'upper_Multi', 'p_Multi']

# Fill null values and save the final table
i = i.fillna('')
i.to_csv('results/unimulti.csv', index=False)

# Print a message
print(f"Table saved; the multivariable analysis included {len(selected_features)}  manually specified variables.")
print(f"CSV data saved; the full sample contains {len(multi_data)} rows.")

i

#### Output the HR of each feature

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, train_data.shape[1]-4))
cph.plot(hazard_ratios=True)
plt.savefig('img/Clinic_feature_pvalue.svg')
plt.show()

# KM curves

Group by HR and compute KM and the log-rank test

In [ ]:
from lifelines import CoxPHFitter
from lifelines.statistics import logrank_test
from lifelines import KaplanMeierFitter
from lifelines.plotting import add_at_risk_counts

# Determine the optimal cutoff using the log-rank maximization method (computed on the training set)
def find_optimal_cutoff_logrank(risk_scores, durations, events, min_group_ratio=0.1):
    """
    Find the optimal cutoff using the log-rank statistic maximization method
    The principle is consistent with X-tile software, but without multiple-comparison correction
    """
    n = len(risk_scores)
    min_size = int(n * min_group_ratio)
    
    # Candidate cutoffs: 10%-90% quantiles
    percentiles = np.linspace(np.percentile(risk_scores, 10), 
                              np.percentile(risk_scores, 90), 100)
    
    best_stat = -np.inf
    best_cutoff = np.median(risk_scores)  # default value
    
    for cutoff in percentiles:
        high_mask = risk_scores > cutoff
        low_mask = risk_scores <= cutoff
        
        # Ensure both groups have enough samples
        if sum(high_mask) < min_size or sum(low_mask) < min_size:
            continue
        # Ensure both groups have events
        if events[high_mask].sum() < 3 or events[low_mask].sum() < 3:
            continue
        
        try:
            result = logrank_test(
                durations[high_mask], durations[low_mask],
                events[high_mask], events[low_mask]
            )
            if result.test_statistic > best_stat:
                best_stat = result.test_statistic
                best_cutoff = cutoff
        except:
            continue
    
    return best_cutoff

# Get the list of features actually used by the model
model_features = list(cph.params_.index)
print(f"Model features: {model_features}")

# Use the raw data (containing all features) instead of the univariable-filtered train_data
train_data_full = data[data[group_info] == 'train']

# Determine the optimal cutoff on the training set
train_risk = cph.predict_partial_hazard(train_data_full[model_features])
optimal_cutoff = find_optimal_cutoff_logrank(
    train_risk.values.flatten(),
    train_data_full[duration_col].values,
    train_data_full[event_col].values
)
print(f"\nOptimal cutoff on the training set (log-rank maximization): {optimal_cutoff:.4f}")

thres = 1e-4
bst_split = {'train': 1.0, 'val':1, 'test': 1}
loc = {'train': 0.5, 'val':0.5, 'test': 0.5}

for subset, test_data in val_datasets.items():
    # Ensure the test data contains the features required by the model
    test_subset = data[data[group_info] == subset]
    c_index = cph.score(test_subset[model_features + [duration_col, event_col]], scoring_method="concordance_index")
    y_pred = cph.predict_partial_hazard(test_subset[model_features])
    cox_data = pd.concat([test_subset.reset_index(drop=True), y_pred.reset_index(drop=True)], axis=1)
    # Use the optimal cutoff determined on the training set
    cox_data['HR'] = cox_data[0] > optimal_cutoff

    dem = (cox_data["HR"] == True)
    results = logrank_test(cox_data[duration_col][dem], cox_data[duration_col][~dem], 
                           event_observed_A=cox_data[event_col][dem], event_observed_B=cox_data[event_col][~dem])
    p_value = f"={results.p_value:.3f}" if results.p_value > thres else f'<{thres}'
    plt.title(f"Cohort {subset} C-index:{c_index:.3f}")
    plt.ylabel('Probability')
    if sum(dem):
        kmf_high = KaplanMeierFitter()
        kmf_high.fit(cox_data[duration_col][dem], event_observed=cox_data[event_col][dem], label="High Risk")
        kmf_high.plot_survival_function(color='r')
    if sum(~dem):
        kmf_low = KaplanMeierFitter()
        kmf_low.fit(cox_data[duration_col][~dem], event_observed=cox_data[event_col][~dem], label="Low Risk")
        kmf_low.plot_survival_function(color='g')
    plt.text(0.5, loc[subset] if subset in loc else 0.2, f"P{p_value}")
    plt.xlabel('Time(months)')
    add_at_risk_counts(kmf_high, kmf_low, rows_to_show=['At risk'])
    plt.savefig(f'img/{task_type}KM_{subset}.svg', bbox_inches='tight')
    plt.show()

In [ ]:
import os
import numpy as np

def get_prediction(model: CoxPHFitter, data, ID=None, **kwargs):
    hr = model.predict_partial_hazard(data)
    expectation = model.predict_expectation(data)
    
    predictions = pd.concat([hr, expectation], axis=1)
    predictions.columns = ['HR', 'expectation']
    if ID is not None:
        predictions = pd.concat([ID, hr, expectation], axis=1)
        predictions.columns = ['ID', 'HR', 'expectation']
    else:
        predictions = pd.concat([hr, expectation], axis=1)
        predictions.columns = ['HR', 'expectation']
    return predictions

os.makedirs('results', exist_ok=True)
info = []
for subset, test_data in val_datasets.items():
    if subset in get_param_in_cwd('subsets'):
        results = get_prediction(cph, test_data, ID=test_data['ID'])
        results.to_csv(f'results/{task_type}cox_predictions_{subset}.csv', index=False)
        results['group'] = subset
        info.append(results)
        pd.merge(results, label_data[['ID', event_col, duration_col]], on='ID', how='inner').to_csv(f'features/{task_type}4xtile_{subset}.txt', 
                                                                                                    index=False, sep='\t')
info = pd.concat(info, axis=0)
info